# P121 — MobileNets: redes convolucionales eficientes para visión en dispositivos móviles

## 1. Título y paper

**Paper:** *MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications*  
**Autoría:** Andrew G. Howard, Menglong Zhu, Bo Chen, Dmitry Kalenichenko, Weijun Wang, Tobias Weyand, Marco Andreetto, Hartwig Adam  
**Año y venue:** 2017 · arXiv:1704.04861  
**Nivel:** L2 · **Motor:** `mobilenets`  
**Ficha completa:** [`P121_mobilenets`](../../papers/foundational/P121_mobilenets/README.md)

**Hito:** Descompone la convolución en dos pasos y convierte el compromiso entre precisión y coste en dos perillas explícitas que el ingeniero elige.

- [arXiv:1704.04861](https://arxiv.org/abs/1704.04861)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Las redes de visión que funcionaban exigían un centro de datos. En un teléfono, un sensor o un vehículo, el presupuesto es de milivatios y milisegundos, y no había forma sistemática de elegir dónde recortar.
2. Ejecutar una implementación mínima de la propuesta: Convolución separable en profundidad —filtrar cada canal por separado y luego combinarlos con núcleos de 1×1—, más un multiplicador de anchura y otro de resolución que parametrizan la familia entera.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P04
- P45


## 4. Intuición

Una convolución hace dos cosas a la vez: filtrar el espacio y mezclar canales. Separarlas en dos pasos cuesta casi nueve veces menos y produce casi lo mismo.


## 5. Concepto mínimo

```text
Estándar   : k² · M · N · S²
Separable  : k² · M · S²  +  M · N · S²

    razón = 1/N + 1/k²      ← con k=3, el techo es 9×
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('mobilenets', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto ahorra la convolución separable en la red completa?
2. ¿Coincide con la fórmula?
3. ¿Cómo escala el multiplicador de anchura?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('mobilenets', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('mobilenets', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

**935,7 millones** de multiplicaciones con convolución estándar frente a **111,1** con separable: **8,42×**. La fórmula 1/N + 1/k² predice **0,1131** para 512 canales y la capa medida da **0,1131**. Con α = 0,5 el coste baja a **0,265** del total; con α = 0,25, a **0,075** — aproximadamente el cuadrado de α.


## 10. Comentario pedagógico

Fíjate en que el término dominante es 1/k², no 1/N. El ahorro se satura cerca de 9× por mucho que crezcan los canales, así que la separación resuelve un factor constante, no un orden de magnitud creciente. Quien necesita más recorta con las otras dos perillas: anchura y resolución.


## 11. Error o anti-patrón deliberado

Anti-patrón: dar por hecho que menos operaciones significa menos tiempo.


In [ ]:
print('Se cuentan multiplicaciones, no milisegundos.')
print('La convolucion en profundidad aprovecha mucho peor la memoria que la estandar.')
print('Un modelo con 9x menos operaciones puede ir solo 3x mas rapido en hardware real.')

## 12. Corrección

Las dos perillas, por separado:


In [ ]:
r = run_paper_lab('mobilenets', seed=3)['result']
print('razon global:', r['razon_global'])
for fila in r['por_capa']:
    print('  ', fila)
print('multiplicador de anchura:')
for fila in r['multiplicador_de_anchura']:
    print('  ', fila)

## 13. Desafío guiado

Explica por qué el ahorro tiene un techo y qué habría que cambiar para superarlo.


In [ ]:
r = run_paper_lab('mobilenets', seed=3)['result']
show(r)

## 14. Desafío autónomo

Toma un modelo que despliegues y calcula sus multiplicaciones-acumulaciones. Estima qué ahorrarías separando sus convoluciones, y contrástalo con una medición de tiempo real.


## 15. Evidencia de aprendizaje

Guarda el cálculo y la diferencia entre el ahorro teórico y el medido.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P121_mobilenets/README.md) · evaluación formal: [`assessments/papers/P121_mobilenets.md`](../../assessments/papers/P121_mobilenets.md)


## 16. Cierre

El cómputo cabe en el dispositivo. Falta que el modelo entienda documentos, donde la posición es parte del significado.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
